In [1]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/09 15:21:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
import pandas as pd

In [4]:
df_pandas = pd.read_parquet('data/pq/yellow/2025/11/yellow_tripdata_2025-11.parquet')

In [5]:
df_pandas.dtypes

VendorID                          int32
tpep_pickup_datetime     datetime64[us]
tpep_dropoff_datetime    datetime64[us]
passenger_count                 float64
trip_distance                   float64
RatecodeID                      float64
store_and_fwd_flag               object
PULocationID                      int32
DOLocationID                      int32
payment_type                      int64
fare_amount                     float64
extra                           float64
mta_tax                         float64
tip_amount                      float64
tolls_amount                    float64
improvement_surcharge           float64
total_amount                    float64
congestion_surcharge            float64
Airport_fee                     float64
cbd_congestion_fee              float64
dtype: object

In [2]:
from pyspark.sql import types

In [3]:
schema = types.StructType(
    [
        types.StructField("VendorID", types.IntegerType(), True),
        types.StructField("tpep_pickup_datetime", types.TimestampNTZType(), True),
        types.StructField("tpep_dropoff_datetime", types.TimestampNTZType(), True),
        types.StructField("passenger_count", types.LongType(), True),
        types.StructField("trip_distance", types.DoubleType(), True),
        types.StructField("RatecodeID", types.LongType(), True),
        types.StructField("store_and_fwd_flag", types.StringType(), True),
        types.StructField("PULocationID", types.IntegerType(), True),
        types.StructField("DOLocationID", types.IntegerType(), True),
        types.StructField("payment_type", types.LongType(), True),
        types.StructField("fare_amount", types.DoubleType(), True),
        types.StructField("extra", types.DoubleType(), True),
        types.StructField("mta_tax", types.DoubleType(), True),
        types.StructField("tip_amount", types.DoubleType(), True),
        types.StructField("tolls_amount", types.DoubleType(), True),
        types.StructField("improvement_surcharge", types.DoubleType(), True),
        types.StructField("total_amount", types.DoubleType(), True),
        types.StructField("congestion_surcharge", types.DoubleType(), True),
        types.StructField("Airport_fee", types.DoubleType(), True),
        types.StructField("cbd_congestion_fee", types.DoubleType(), True),
    ]
)

In [4]:
df_yellow_2025_11 = spark.read \
    .schema(schema) \
    .parquet('data/pq/yellow/2025/11/yellow_tripdata_2025-11.parquet')

In [5]:
df_yellow_2025_11.show()

[Stage 0:>                                                          (0 + 1) / 1]

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|              1|         1.68|         1|                 N|          43|    

In [9]:
df_yellow_2025_11.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [10]:
df_yellow_2025_11 \
    .repartition(4) \
    .write.parquet('data/homework/', mode='overwrite')

In [6]:
df_yellow_2025_11.createOrReplaceTempView('yellow202511')

In [12]:
df_yellow_2025_11.columns

['VendorID',
 'tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'RatecodeID',
 'store_and_fwd_flag',
 'PULocationID',
 'DOLocationID',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'congestion_surcharge',
 'Airport_fee',
 'cbd_congestion_fee']

In [8]:
df_yellow_quest3 = spark.sql("""
SELECT
    COUNT(*) AS number_records
FROM
    yellow202511
WHERE
    date_trunc('day', tpep_pickup_datetime) = '2025-11-15 00:00:00'
""")

In [9]:
df_yellow_quest3.show()

[Stage 1:=============================>                             (1 + 1) / 2]

+--------------+
|number_records|
+--------------+
|        162604|
+--------------+



In [10]:
from pyspark.sql import functions as F

In [11]:
df_yellow_quest3 = df_yellow_2025_11 \
    .filter(F.to_date('tpep_pickup_datetime') == '2025-11-15') \
    .select(F.count('*').alias('number_records'))

In [10]:
df_yellow_quest3.show()

+--------------+
|number_records|
+--------------+
|        162604|
+--------------+



In [34]:
df_yellow_quest4 = spark.sql("""
SELECT 
    MAX(ROUND((unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)) / 60 / 60, 1)) AS longest_trip
FROM
    yellow202511
""")

In [35]:
df_yellow_quest4.show()

[Stage 23:>                                                         (0 + 2) / 2]

+------------+
|longest_trip|
+------------+
|        90.6|
+------------+



In [16]:
df_zones = spark.read.parquet('zones/')

In [17]:
df_zones.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [19]:
df_yellow_quest6 = df_yellow_2025_11 \
    .join(df_zones, df_yellow_2025_11.PULocationID == df_zones.LocationID)

In [20]:
df_yellow_quest6.createOrReplaceTempView('yellow_with_zones')

In [41]:
df_yellow_quest6 = spark.sql("""
SELECT
    Zone,
    COUNT(*) AS pickup_count
FROM
    yellow_with_zones
GROUP BY Zone
ORDER BY pickup_count
LIMIT 10;
""")

In [42]:
df_yellow_quest6.show(truncate=False)

[Stage 46:>                                                         (0 + 2) / 2]

+---------------------------------------------+------------+
|Zone                                         |pickup_count|
+---------------------------------------------+------------+
|Governor's Island/Ellis Island/Liberty Island|1           |
|Eltingville/Annadale/Prince's Bay            |1           |
|Arden Heights                                |1           |
|Port Richmond                                |3           |
|Rikers Island                                |4           |
|Rossville/Woodrow                            |4           |
|Great Kills                                  |4           |
|Green-Wood Cemetery                          |4           |
|Jamaica Bay                                  |5           |
|Westerleigh                                  |12          |
+---------------------------------------------+------------+

